# Collect Data

In [58]:
%pip install pandas numpy seaborn matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [59]:
import pandas as pd
import numpy as np


In [60]:
import time
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [61]:
df = pd.read_csv('final_data_test.csv')
df['Periodo'] = pd.to_datetime(df['Periodo'], format='%Y-%m-%d')
df

,Nome_Poco_ANP,Periodo,Estado,Bacia,Petroleo,Agua,Gas,Tempo_Producao,Grau_API,Densidade_GLP_Gas,Densidade_GLP_Liquido,PCS,Latitude,Longitude,Profundidade,Lamina_DAgua,Campo,Idade,Preco_referencia
0,7-AB-103HP-RJS,2018-01-31,Rio de Janeiro,Campos,1369.6102,15.4729,8.8017,677.0,26.2,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2774,1210.1772
1,7-AB-103HP-RJS,2018-05-31,Rio de Janeiro,Campos,1387.1269,14.8405,12.3111,725.0,26.2,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2894,1538.6618
2,7-AB-103HP-RJS,2018-06-30,Rio de Janeiro,Campos,1293.5024,30.2712,12.9338,720.0,26.2,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2924,1562.2000
3,7-AB-103HP-RJS,2018-07-31,Rio de Janeiro,Campos,740.0016,11.5824,8.2655,484.0,26.2,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2955,1600.4596
4,7-AB-103HP-RJS,2018-09-30,Rio de Janeiro,Campos,611.2813,2.4729,7.1527,332.0,26.2,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,3016,1807.4609
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29882,7-XA-4D-CES,2019-06-30,Ceará,Ceará,94.1782,251.5236,0.2339,717.0,28.1,2.039707,532.790002,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14097,1424.4825
29883,7-XA-4D-CES,2019-07-31,Ceará,Ceará,93.5297,167.4319,0.3863,742.0,28.1,2.039707,532.790002,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14128,1388.7033
29884,7-XA-4D-CES,2019-08-31,Ceará,Ceará,88.4686,141.8379,0.2658,737.0,28.1,2.039707,532.790002,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14159,1287.9177
29885,7-XA-4D-CES,2019-09-30,Ceará,Ceará,103.8475,95.3443,0.2822,709.0,28.1,2.039707,532.790002,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14189,1455.1776


In [62]:
df = df[df['Petroleo'] > 0]
df = df[df['Agua'] > 0]
df = df[df['Gas'] > 0]
df = df[df['Lamina_DAgua'] > 0]
df

,Nome_Poco_ANP,Periodo,Estado,Bacia,Petroleo,Agua,Gas,Tempo_Producao,Grau_API,Densidade_GLP_Gas,Densidade_GLP_Liquido,PCS,Latitude,Longitude,Profundidade,Lamina_DAgua,Campo,Idade,Preco_referencia
0,7-AB-103HP-RJS,2018-01-31,Rio de Janeiro,Campos,1369.6102,15.4729,8.8017,677.0,26.2,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2774,1210.1772
1,7-AB-103HP-RJS,2018-05-31,Rio de Janeiro,Campos,1387.1269,14.8405,12.3111,725.0,26.2,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2894,1538.6618
2,7-AB-103HP-RJS,2018-06-30,Rio de Janeiro,Campos,1293.5024,30.2712,12.9338,720.0,26.2,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2924,1562.2000
3,7-AB-103HP-RJS,2018-07-31,Rio de Janeiro,Campos,740.0016,11.5824,8.2655,484.0,26.2,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2955,1600.4596
4,7-AB-103HP-RJS,2018-09-30,Rio de Janeiro,Campos,611.2813,2.4729,7.1527,332.0,26.2,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,3016,1807.4609
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29882,7-XA-4D-CES,2019-06-30,Ceará,Ceará,94.1782,251.5236,0.2339,717.0,28.1,2.039707,532.790002,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14097,1424.4825
29883,7-XA-4D-CES,2019-07-31,Ceará,Ceará,93.5297,167.4319,0.3863,742.0,28.1,2.039707,532.790002,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14128,1388.7033
29884,7-XA-4D-CES,2019-08-31,Ceará,Ceará,88.4686,141.8379,0.2658,737.0,28.1,2.039707,532.790002,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14159,1287.9177
29885,7-XA-4D-CES,2019-09-30,Ceará,Ceará,103.8475,95.3443,0.2822,709.0,28.1,2.039707,532.790002,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14189,1455.1776


In [63]:
df = df[df['Estado'] == 'Rio de Janeiro']
df = df[df['Bacia'] == 'Campos']
df

,Nome_Poco_ANP,Periodo,Estado,Bacia,Petroleo,Agua,Gas,Tempo_Producao,Grau_API,Densidade_GLP_Gas,Densidade_GLP_Liquido,PCS,Latitude,Longitude,Profundidade,Lamina_DAgua,Campo,Idade,Preco_referencia
0,7-AB-103HP-RJS,2018-01-31,Rio de Janeiro,Campos,1369.6102,15.4729,8.8017,677.0,26.20,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2774,1210.1772
1,7-AB-103HP-RJS,2018-05-31,Rio de Janeiro,Campos,1387.1269,14.8405,12.3111,725.0,26.20,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2894,1538.6618
2,7-AB-103HP-RJS,2018-06-30,Rio de Janeiro,Campos,1293.5024,30.2712,12.9338,720.0,26.20,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2924,1562.2000
3,7-AB-103HP-RJS,2018-07-31,Rio de Janeiro,Campos,740.0016,11.5824,8.2655,484.0,26.20,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2955,1600.4596
4,7-AB-103HP-RJS,2018-09-30,Rio de Janeiro,Campos,611.2813,2.4729,7.1527,332.0,26.20,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,3016,1807.4609
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29853,7-VM-14D-RJS,2025-08-31,Rio de Janeiro,Campos,123.0542,2475.7504,0.4162,432.0,22.86,2.044230,533.350020,38122.0594,-22.189655,-40.303273,0.0,81.0,VERMELHO,14540,2048.8904
29854,7-VM-14D-RJS,2025-09-30,Rio de Janeiro,Campos,99.5146,2192.6278,0.4992,408.0,22.86,2.044230,533.350020,38122.0594,-22.189655,-40.303273,0.0,81.0,VERMELHO,14570,2005.4093
29855,7-VM-14D-RJS,2025-10-31,Rio de Janeiro,Campos,67.0630,1098.4849,0.7693,230.0,22.86,2.044230,533.350020,38122.0594,-22.189655,-40.303273,0.0,81.0,VERMELHO,14601,1926.1265
29856,7-VM-14D-RJS,2025-11-30,Rio de Janeiro,Campos,136.2777,2616.5610,1.2561,481.0,22.86,2.044230,533.350020,38122.0594,-22.189655,-40.303273,0.0,81.0,VERMELHO,14631,1820.0921


In [64]:
from sklearn.ensemble import IsolationForest

features = df.drop(columns=['Nome_Poco_ANP', 'Periodo', 'Estado', 'Bacia', 'Latitude', 'Longitude', 'Campo', 'Profundidade'])
features = features.dropna()

iso_forest = IsolationForest(random_state=42)
iso_forest.fit(features)

df = df.loc[features.index].copy()
df['anomaly_score'] = iso_forest.decision_function(features)
df['anomaly'] = iso_forest.predict(features)

df

,Nome_Poco_ANP,Periodo,Estado,Bacia,Petroleo,Agua,Gas,Tempo_Producao,Grau_API,Densidade_GLP_Gas,...,PCS,Latitude,Longitude,Profundidade,Lamina_DAgua,Campo,Idade,Preco_referencia,anomaly_score,anomaly
0,7-AB-103HP-RJS,2018-01-31,Rio de Janeiro,Campos,1369.6102,15.4729,8.8017,677.0,26.20,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2774,1210.1772,0.037952,1
1,7-AB-103HP-RJS,2018-05-31,Rio de Janeiro,Campos,1387.1269,14.8405,12.3111,725.0,26.20,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2894,1538.6618,0.043747,1
2,7-AB-103HP-RJS,2018-06-30,Rio de Janeiro,Campos,1293.5024,30.2712,12.9338,720.0,26.20,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2924,1562.2000,0.043747,1
3,7-AB-103HP-RJS,2018-07-31,Rio de Janeiro,Campos,740.0016,11.5824,8.2655,484.0,26.20,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2955,1600.4596,0.037014,1
4,7-AB-103HP-RJS,2018-09-30,Rio de Janeiro,Campos,611.2813,2.4729,7.1527,332.0,26.20,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,3016,1807.4609,0.037483,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29853,7-VM-14D-RJS,2025-08-31,Rio de Janeiro,Campos,123.0542,2475.7504,0.4162,432.0,22.86,2.044230,...,38122.0594,-22.189655,-40.303273,0.0,81.0,VERMELHO,14540,2048.8904,-0.022612,-1
29854,7-VM-14D-RJS,2025-09-30,Rio de Janeiro,Campos,99.5146,2192.6278,0.4992,408.0,22.86,2.044230,...,38122.0594,-22.189655,-40.303273,0.0,81.0,VERMELHO,14570,2005.4093,-0.025901,-1
29855,7-VM-14D-RJS,2025-10-31,Rio de Janeiro,Campos,67.0630,1098.4849,0.7693,230.0,22.86,2.044230,...,38122.0594,-22.189655,-40.303273,0.0,81.0,VERMELHO,14601,1926.1265,-0.044298,-1
29856,7-VM-14D-RJS,2025-11-30,Rio de Janeiro,Campos,136.2777,2616.5610,1.2561,481.0,22.86,2.044230,...,38122.0594,-22.189655,-40.303273,0.0,81.0,VERMELHO,14631,1820.0921,-0.021413,-1


In [65]:
sample_df = df[df['anomaly'] == 1]
sample_df

,Nome_Poco_ANP,Periodo,Estado,Bacia,Petroleo,Agua,Gas,Tempo_Producao,Grau_API,Densidade_GLP_Gas,...,PCS,Latitude,Longitude,Profundidade,Lamina_DAgua,Campo,Idade,Preco_referencia,anomaly_score,anomaly
0,7-AB-103HP-RJS,2018-01-31,Rio de Janeiro,Campos,1369.6102,15.4729,8.8017,677.0,26.2,2.007566,...,41614.26010,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2774,1210.1772,0.037952,1
1,7-AB-103HP-RJS,2018-05-31,Rio de Janeiro,Campos,1387.1269,14.8405,12.3111,725.0,26.2,2.007566,...,41614.26010,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2894,1538.6618,0.043747,1
2,7-AB-103HP-RJS,2018-06-30,Rio de Janeiro,Campos,1293.5024,30.2712,12.9338,720.0,26.2,2.007566,...,41614.26010,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2924,1562.2000,0.043747,1
3,7-AB-103HP-RJS,2018-07-31,Rio de Janeiro,Campos,740.0016,11.5824,8.2655,484.0,26.2,2.007566,...,41614.26010,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2955,1600.4596,0.037014,1
4,7-AB-103HP-RJS,2018-09-30,Rio de Janeiro,Campos,611.2813,2.4729,7.1527,332.0,26.2,2.007566,...,41614.26010,-22.031677,-39.943771,2623.5,626.0,ALBACORA,3016,1807.4609,0.037483,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29483,7-TVD-8D-RJS,2025-08-31,Rio de Janeiro,Campos,3072.1276,1629.6683,23.6395,598.0,26.8,1.987930,...,42385.04465,-22.983486,-40.735555,-2982.6,910.0,TARTARUGA VERDE,792,2148.5860,0.005286,1
29484,7-TVD-8D-RJS,2025-09-30,Rio de Janeiro,Campos,2824.0924,1555.7035,21.2706,587.0,26.8,1.987930,...,42385.04465,-22.983486,-40.735555,-2982.6,910.0,TARTARUGA VERDE,822,2106.3811,0.009034,1
29485,7-TVD-8D-RJS,2025-10-31,Rio de Janeiro,Campos,2915.1964,1605.8897,20.1178,611.0,26.8,1.987930,...,42385.04465,-22.983486,-40.735555,-2982.6,910.0,TARTARUGA VERDE,853,2016.3880,0.006435,1
29486,7-TVD-8D-RJS,2025-11-30,Rio de Janeiro,Campos,3009.8116,1658.0103,19.2372,584.0,26.8,1.987930,...,42385.04465,-22.983486,-40.735555,-2982.6,910.0,TARTARUGA VERDE,883,1930.0290,0.003925,1


In [66]:
sample_df['Petroleo'].describe()

count    12915.000000
mean      2481.806636
std       2025.733086
min          0.010900
25%       1095.432150
50%       1971.685600
75%       3209.500050
max      14500.866400
Name: Petroleo, dtype: float64

In [67]:
target_column = 'Petroleo'
ignore_columns = [
    'Nome_Poco_ANP', 'Latitude', 'Longitude', 'Campo', 'Estado', 'Bacia', 'Periodo', 'Profundidade'
]

In [68]:
# sample_df = df.sample(frac=1, random_state=42)
# sample_df.drop_duplicates(inplace=True)
# columns = [col for col in sample_df.columns if col not in ignore_columns]
# for col in columns:
#     Q1 = sample_df[col].quantile(0.25)
#     Q3 = sample_df[col].quantile(0.75)
#     IQR = Q3 - Q1

#     lower_bound = Q1 - 1.5 * IQR
#     upper_bound = Q3 + 1.5 * IQR

#     sample_df = sample_df[(sample_df[col] >= lower_bound) & (sample_df[col] <= upper_bound)]

# sample_df

In [69]:
train = sample_df[sample_df['Periodo'] < pd.to_datetime('2024-01-01')]
test = sample_df[sample_df['Periodo'] >= pd.to_datetime('2024-01-01')]

In [70]:
train

,Nome_Poco_ANP,Periodo,Estado,Bacia,Petroleo,Agua,Gas,Tempo_Producao,Grau_API,Densidade_GLP_Gas,...,PCS,Latitude,Longitude,Profundidade,Lamina_DAgua,Campo,Idade,Preco_referencia,anomaly_score,anomaly
0,7-AB-103HP-RJS,2018-01-31,Rio de Janeiro,Campos,1369.6102,15.4729,8.8017,677.0,26.2,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2774,1210.1772,0.037952,1
1,7-AB-103HP-RJS,2018-05-31,Rio de Janeiro,Campos,1387.1269,14.8405,12.3111,725.0,26.2,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2894,1538.6618,0.043747,1
2,7-AB-103HP-RJS,2018-06-30,Rio de Janeiro,Campos,1293.5024,30.2712,12.9338,720.0,26.2,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2924,1562.2000,0.043747,1
3,7-AB-103HP-RJS,2018-07-31,Rio de Janeiro,Campos,740.0016,11.5824,8.2655,484.0,26.2,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2955,1600.4596,0.037014,1
4,7-AB-103HP-RJS,2018-09-30,Rio de Janeiro,Campos,611.2813,2.4729,7.1527,332.0,26.2,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,3016,1807.4609,0.037483,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29432,7-TVD-5D-RJS,2023-08-31,Rio de Janeiro,Campos,7455.4895,1092.3278,76.1519,744.0,26.8,1.997850,...,41801.2270,-22.962363,-40.709910,-3069.6,941.0,TARTARUGA VERDE,1231,2419.4681,0.016739,1
29433,7-TVD-5D-RJS,2023-09-30,Rio de Janeiro,Campos,7327.0160,1084.6882,76.5751,718.0,26.8,1.997850,...,41801.2270,-22.962363,-40.709910,-3069.6,941.0,TARTARUGA VERDE,1261,2700.6249,0.011067,1
29434,7-TVD-5D-RJS,2023-10-31,Rio de Janeiro,Campos,5952.7208,1687.2829,65.3762,684.0,26.8,1.997850,...,41801.2270,-22.962363,-40.709910,-3069.6,941.0,TARTARUGA VERDE,1292,2677.4448,0.017562,1
29435,7-TVD-5D-RJS,2023-11-30,Rio de Janeiro,Campos,5987.0127,1826.6325,73.8396,720.0,26.8,2.007620,...,42550.6580,-22.962363,-40.709910,-3069.6,941.0,TARTARUGA VERDE,1322,2349.3362,0.016787,1


In [71]:
test

,Nome_Poco_ANP,Periodo,Estado,Bacia,Petroleo,Agua,Gas,Tempo_Producao,Grau_API,Densidade_GLP_Gas,...,PCS,Latitude,Longitude,Profundidade,Lamina_DAgua,Campo,Idade,Preco_referencia,anomaly_score,anomaly
275,7-AB-134HPA-RJS,2024-01-31,Rio de Janeiro,Campos,10981.5097,766.5725,136.0015,723.0,22.85,2.066945,...,36846.73540,-22.066130,-39.872239,-2536.23,1015.0,ALBACORA,1938,2091.9442,0.014238,1
276,7-AB-134HPA-RJS,2024-02-29,Rio de Janeiro,Campos,11732.4494,1108.8803,149.3063,687.0,22.85,2.066945,...,36846.73540,-22.066130,-39.872239,-2536.23,1015.0,ALBACORA,1967,2184.2019,0.004795,1
277,7-AB-134HPA-RJS,2024-03-31,Rio de Janeiro,Campos,11073.3941,1046.5905,156.7299,742.0,22.85,2.066945,...,36846.73540,-22.066130,-39.872239,-2536.23,1015.0,ALBACORA,1998,2219.8154,0.009697,1
278,7-AB-134HPA-RJS,2024-04-30,Rio de Janeiro,Campos,10836.5495,1707.7264,151.6220,720.0,22.85,2.066945,...,36846.73540,-22.066130,-39.872239,-2536.23,1015.0,ALBACORA,2028,2428.1163,0.005670,1
279,7-AB-134HPA-RJS,2024-05-31,Rio de Janeiro,Campos,10376.2290,1649.5664,143.3321,744.0,22.85,2.066945,...,36846.73540,-22.066130,-39.872239,-2536.23,1015.0,ALBACORA,2059,2246.6465,0.014869,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29483,7-TVD-8D-RJS,2025-08-31,Rio de Janeiro,Campos,3072.1276,1629.6683,23.6395,598.0,26.80,1.987930,...,42385.04465,-22.983486,-40.735555,-2982.60,910.0,TARTARUGA VERDE,792,2148.5860,0.005286,1
29484,7-TVD-8D-RJS,2025-09-30,Rio de Janeiro,Campos,2824.0924,1555.7035,21.2706,587.0,26.80,1.987930,...,42385.04465,-22.983486,-40.735555,-2982.60,910.0,TARTARUGA VERDE,822,2106.3811,0.009034,1
29485,7-TVD-8D-RJS,2025-10-31,Rio de Janeiro,Campos,2915.1964,1605.8897,20.1178,611.0,26.80,1.987930,...,42385.04465,-22.983486,-40.735555,-2982.60,910.0,TARTARUGA VERDE,853,2016.3880,0.006435,1
29486,7-TVD-8D-RJS,2025-11-30,Rio de Janeiro,Campos,3009.8116,1658.0103,19.2372,584.0,26.80,1.987930,...,42385.04465,-22.983486,-40.735555,-2982.60,910.0,TARTARUGA VERDE,883,1930.0290,0.003925,1


In [72]:
X_train = train.drop([target_column] + ignore_columns, axis=1)
X_test = test.drop([target_column] + ignore_columns, axis=1)

y_train = train[target_column]
y_test = test[target_column]

In [73]:
# train = sample_df[~sample_df['Nome_Poco_ANP'].isin(pocos_teste)]
# test = sample_df[sample_df['Nome_Poco_ANP'].isin(pocos_teste)]

# X_train = train.drop([target_column] + ignore_columns, axis=1)
# y_train = train[target_column]

# X_test = test.drop([target_column] + ignore_columns, axis=1)
# y_test = test[target_column]

In [74]:
# X = sample_df.drop(ignore_columns + [target_column], axis=1)
# y = sample_df[target_column]
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.18, random_state=42)

In [75]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Evaluate Methods

In [76]:
results_df = pd.DataFrame()

def evaluate_and_record(model_name, model, X_train, y_train, X_test, y_test, best_params=None):
    global results_df
    
    start_time = time.time()
    model.fit(X_train, y_train)
    end_time = time.time()
    training_time = end_time - start_time
    
    y_pred_treino = model.predict(X_train)
    y_pred_teste = model.predict(X_test)
    
    rmse_treino = np.sqrt(mean_squared_error(y_train, y_pred_treino))
    r2_treino = r2_score(y_train, y_pred_treino)
    mae_treino = mean_absolute_error(y_train, y_pred_treino)
    
    rmse_teste = np.sqrt(mean_squared_error(y_test, y_pred_teste))
    r2_teste = r2_score(y_test, y_pred_teste)
    mae_teste = mean_absolute_error(y_test, y_pred_teste)
    
    new_row = {
        'Modelo': model_name,
        'R2 Treino': r2_treino,
        'R2 Teste': r2_teste,
        'RMSE Treino': rmse_treino,
        'RMSE Teste': rmse_teste,
        'MAE Treino': mae_treino,
        'MAE Teste': mae_teste,
        'Tempo (segundos)': training_time,
        'Hyperparametros': str(best_params) if best_params else 'N/A'
    }
    
    results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)
    return model

In [77]:
def record_gridsearch_details(grid_search_object, model_prefix):
    global results_df
    
    results = pd.DataFrame(grid_search_object.cv_results_)
    
    for index, row in results.iterrows():
        mean_r2_cv = row['mean_test_score']
        
        mean_fit_time = row['mean_fit_time']
        
        model_name_detail = f"{model_prefix} - Combinacao {index + 1}"
        
        new_row = {
            'Modelo': model_name_detail,
            'R2 Teste': mean_r2_cv,
            'RMSE Teste': np.nan,
            'Tempo (s)': mean_fit_time,
            'Detalhes': f"Combin. Testada (Rank: {row['rank_test_score']})",
            'Hyperparametros': str(row['params'])
        }
        
        results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)

In [78]:
cv_strategy = KFold(n_splits=3, shuffle=True, random_state=42)

# LinearRegression

In [79]:
model = LinearRegression()
evaluate_and_record("Regressao Linear", model, X_train_scaled, y_train, X_test_scaled, y_test)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](12,)","[ -84.84,1314.41, 451.97,..., 6.47,-260.99, 0. ]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,2494
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,12
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(11)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](12,)","[164.94,140.05,115.32,..., 41.18, 0.45, 0. ]"


# Ridge

In [ ]:
scoring_metric = 'neg_mean_squared_error'

ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(random_state=42))
])

alphas = np.logspace(-4, 2, 30)

param_grid_ridge = {
    'ridge__alpha': alphas
}

# 4. GridSearchCV
grid_search_ridge = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=param_grid_ridge,
    scoring=scoring_metric,
    cv=cv_strategy,
    verbose=1,
    n_jobs=-1
)

grid_search_ridge.fit(X_train, y_train)

best_alpha_ridge = grid_search_ridge.best_params_['ridge__alpha']
best_ridge_model = grid_search_ridge.best_estimator_

Fitting 3 folds for each of 30 candidates, totalling 90 fits


In [ ]:
evaluate_and_record('Ridge Otimo', best_ridge_model, X_train_scaled, y_train, X_test_scaled, y_test, best_alpha_ridge)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('ridge', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,12
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value
"mean_ mean_: ndarray of shape (n_features,) or NoneThe mean value for each feature in the training set.Equal to ``None`` when ``with_mean=False`` and ``with_std=False``.","ndarray[float64](12,)","[ 0., 0., 0.,...,-0.,-0., 0.]"


# Lasso

In [ ]:
lasso_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', Lasso(random_state=42, max_iter=2000))
])

param_grid_lasso = {
    'lasso__alpha': alphas
}

grid_search_lasso = GridSearchCV(
    estimator=lasso_pipeline,
    param_grid=param_grid_lasso,
    scoring=scoring_metric,
    cv=cv_strategy,
    verbose=1,
    n_jobs=-1
)

grid_search_lasso.fit(X_train, y_train)

best_alpha_lasso = grid_search_lasso.best_params_['lasso__alpha']
best_lasso_model = grid_search_lasso.best_estimator_

Fitting 3 folds for each of 30 candidates, totalling 90 fits


d:\Codigos\MMQ_MSC_Icaro\venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.265497e+09, tolerance: 4.275e+06
  model = cd_fast.enet_coordinate_descent(


In [ ]:
evaluate_and_record('Lasso Otimo', best_lasso_model, X_train_scaled, y_train, X_test_scaled, y_test, best_alpha_lasso)

d:\Codigos\MMQ_MSC_Icaro\venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.265497e+09, tolerance: 4.275e+06
  model = cd_fast.enet_coordinate_descent(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('lasso', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,12
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value
"mean_ mean_: ndarray of shape (n_features,) or NoneThe mean value for each feature in the training set.Equal to ``None`` when ``with_mean=False`` and ``with_std=False``.","ndarray[float64](12,)","[ 0., 0., 0.,...,-0.,-0., 0.]"


In [ ]:
results_df

,Modelo,R2 Treino,R2 Teste,RMSE Treino,RMSE Teste,MAE Treino,MAE Teste,Tempo (segundos),Hyperparametros
0,Regressao Linear,0.613189,0.429823,1278.083951,1445.913293,846.477672,836.380280,0.015588,N/A
1,Ridge Otimo,0.613188,0.429992,1278.085337,1445.698087,846.451246,836.045216,0.003647,0.011721022975334805
2,Lasso Otimo,0.612953,0.432343,1278.474618,1442.714244,846.462164,831.211172,2.586621,0.0001


# Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
rf_model = RandomForestRegressor(random_state=42, n_jobs=1)

param_grid_rf = {
    'n_estimators': [300, 500, 1000],
    'min_samples_split': [2, 5, 8],
    'min_samples_leaf': [1, 2, 4],
    'max_depth': [None, 10, 20],
    'bootstrap': [True, False]
}

# 4. GridSearchCV
grid_search_rf = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid_rf,
    scoring=scoring_metric,
    cv=cv_strategy,
    verbose=1,
    n_jobs=-1
)

grid_search_rf.fit(X_train, y_train)

best_rf_model = grid_search_rf.best_estimator_
best_params_rf = grid_search_rf.best_params_

best_n_estimators_rf = best_params_rf['n_estimators']
best_min_samples_split_rf = best_params_rf['min_samples_split']
best_min_samples_leaf_rf = best_params_rf['min_samples_leaf']
best_bootstrap_rf = best_params_rf['bootstrap']

Fitting 3 folds for each of 162 candidates, totalling 486 fits


KeyboardInterrupt: 

In [ ]:
evaluate_and_record('Random Forest Otimo', best_rf_model, X_train_scaled, y_train, X_test_scaled, y_test, grid_search_rf.best_params_)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",1450
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",3
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",3
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease 

# Gradient Boosting

In [ ]:
%pip install xgboost

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   - -------------------------------------- 2.4/48.9 MB 41.9 MB/s eta 0:00:02
   ----- ---------------------------------- 7.3/48.9 MB 27.8 MB/s eta 0:00:02
   ---------- ----------------------------- 13.1/48.9 MB 27.7 MB/s eta 0:00:02
   -------------- ------------------------- 18.1/48.9 MB 26.1 MB/s eta 0:00:02
   ---------------- ----------------------- 20.2/48.9 MB 22.8 MB/s eta 0:00:02
   ------------------ --------------------- 23.1/48.9 MB 21.0 MB/s eta 0:00:02
   ---------------------- ----------------- 27.5/48.9 MB 21.5 MB/s eta 0:00:01
   ---------------------- ----------------- 27.5/48.9 MB 21.5 MB/s eta 0:00:01
   ----------------------- ---------------- 28.6/48.9 MB 16.7 MB/s eta 0:00:02
   ------------------------ --------------- 29.6/48.9 MB 15.8 MB/s eta 0:00:02
   ------------------------ --------------- 30.1/48.9 MB 14.6 MB/s eta 0:00:02
   ------------------------- -------------- 31.7/48.9 MB 13.5 M


[notice] A new release of pip is available: 25.3 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from xgboost import XGBRegressor

In [ ]:
xgb_model = XGBRegressor(
    objective='reg:squarederror',
    eval_metric='rmse',
    random_state=42,
    n_jobs=1
)

param_grid_xgb = {
    'n_estimators': [1000, 2000, 3000],
    'learning_rate': [0.01, 0.025, 0.05],
    'max_depth': [3, 6, 9],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.8, 1.0]
}

grid_search_xgb = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid_xgb,
    scoring=scoring_metric,
    cv=cv_strategy,
    verbose=1,
    n_jobs=-1
)

# 5. Fit com X_train original (sem necessidade de StandardScaler)
grid_search_xgb.fit(X_train, y_train)

# Resultados
best_xgb_model = grid_search_xgb.best_estimator_
best_params_xgb = grid_search_xgb.best_params_

Fitting 3 folds for each of 240 candidates, totalling 720 fits


In [ ]:
evaluate_and_record('Gradient Boosting Otimo', best_xgb_model, X_train_scaled, y_train, X_test_scaled, y_test, grid_search_xgb.best_params_)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


# KNN Regression

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

In [ ]:
knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor())
])

param_grid_knn = {
    'knn__n_neighbors': [3, 5, 7, 9, 11, 15],
    'knn__weights': ['uniform', 'distance'],
    'knn__p': [1, 2]
}

grid_search_knn = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid_knn,
    scoring=scoring_metric,
    cv=cv_strategy,
    verbose=1,
    n_jobs=-1
)

grid_search_knn.fit(X_train, y_train)

best_knn_model = grid_search_knn.best_estimator_
best_params_knn = grid_search_knn.best_params_

best_n_neighbors_knn = best_params_knn['knn__n_neighbors']
best_weights_knn = best_params_knn['knn__weights']
best_p_knn = best_params_knn['knn__p']

Fitting 3 folds for each of 600 candidates, totalling 1800 fits


In [ ]:
evaluate_and_record('KNN Otimo', best_knn_model, X_train_scaled, y_train, X_test_scaled, y_test, grid_search_knn.best_params_)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",11
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Uniform weights are used by default.See the following example for a demonstration of the impact ofdifferent weighting schemes on predictions::ref:`sphx_glr_auto_examples_neighbors_plot_regression.py`.",'distance'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric metric: str, DistanceMetric object or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.If metric is a DistanceMetric object, it will be passed directly tothe underlying computation routines.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"effective_metric_ effective_metric_: str or callableThe distance metric to use. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'
"effective_metric_params_ effective_metric_params_: dictAdditional keyword arguments for the metric function. For most metricswill be same with `metric_params` parameter, but may also contain the`p` parameter value if the `effective_metric_` attribute is set to'minkowski'.",dict,{}


# SVR (Suport Vector Regression)

In [ ]:
from sklearn.svm import SVR

In [ ]:
svr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR())
])

param_grid_svr = [
    {
        'svr__kernel': ['rbf'],
        'svr__C': [0.1, 1, 10, 100, 1000],
        'svr__gamma': ['scale', 'auto'],
        'svr__epsilon': [0.01, 0.1, 0.5, 1.0]
    },
    {
        'svr__kernel': ['linear'],
        'svr__C': [0.1, 1, 10, 100],
        'svr__epsilon': [0.01, 0.1, 0.5, 1.0]
    },
    {
        'svr__kernel': ['poly', 'sigmoid'],
        'svr__C': [0.1, 1, 10, 100],
        'svr__gamma': ['scale', 'auto'],
        'svr__epsilon': [0.01, 0.1, 0.5],
        'svr__coef0': [0.0, 0.1]
    }
]

grid_search_svr = GridSearchCV(
    estimator=svr_pipeline,
    param_grid=param_grid_svr,
    scoring=scoring_metric,
    cv=cv_strategy,
    verbose=1,
    n_jobs=-1
)

grid_search_svr.fit(X_train, y_train)

best_svr_model = grid_search_svr.best_estimator_
best_params_svr = grid_search_svr.best_params_

best_C_svr = best_params_svr.get('svr__C')
best_epsilon_svr = best_params_svr.get('svr__epsilon')
best_kernel_svr = best_params_svr.get('svr__kernel')
best_gamma_svr = best_params_svr.get('svr__gamma')
best_coef0_svr = best_params_svr.get('svr__coef0')

Fitting 3 folds for each of 480 candidates, totalling 1440 fits


In [ ]:
evaluate_and_record('SVR Otimo', best_svr_model, X_train_scaled, y_train, X_test_scaled, y_test, grid_search_svr.best_params_)

,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'auto'
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.The penalty is a squared l2. For an intuitive visualization of theeffects of scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",11000
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-SVR model. It specifies the epsilon-tubewithin which no penalty is associated in the training loss functionwith points predicted within a distance epsilon from the actualvalue. Must be non-negative.",0.5
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1


# Neural Network

In [ ]:
from sklearn.neural_network import MLPRegressor

In [ ]:
mlp_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPRegressor(random_state=42, max_iter=2000))
])

mlp_param_grid = [
    {
        'mlp__hidden_layer_sizes': [(100, 100), (200, 200), (100, 100, 100)],
        'mlp__activation': ['relu', 'tanh'],
        'mlp__solver': ['adam'],
        'mlp__alpha': [0.1, 0.15]
    },
    {
        'mlp__hidden_layer_sizes': [(100, 100), (200, 200)],
        'mlp__activation': ['relu', 'tanh'],
        'mlp__solver': ['lbfgs'],
        'mlp__alpha': [0.1, 0.15]
    },
    {
        'mlp__hidden_layer_sizes': [(100, 100), (200, 200)],
        'mlp__activation': ['relu', 'tanh'],
        'mlp__solver': ['sgd'],
        'mlp__alpha': [0.1, 0.15],
        'mlp__learning_rate': ['constant', 'adaptive']
    }
]

grid_search_mlp = GridSearchCV(
    estimator=mlp_pipeline,
    param_grid=mlp_param_grid,
    scoring=scoring_metric,
    cv=cv_strategy,
    verbose=1,
    n_jobs=1
)

grid_search_mlp.fit(X_train, y_train)

best_mlp_model = grid_search_mlp.best_estimator_
best_params_mlp = grid_search_mlp.best_params_

Fitting 3 folds for each of 162 candidates, totalling 486 fits


d:\Codigos\MMQ_MSC_Icaro\venv\Lib\site-packages\sklearn\model_selection\_validation.py:489: FitFailedWarning: 
48 fits failed out of a total of 486.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
48 fits failed with the following error:
Traceback (most recent call last):
  File "d:\Codigos\MMQ_MSC_Icaro\venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 851, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Codigos\MMQ_MSC_Icaro\venv\Lib\site-packages\sklearn\base.py", line 1403, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "d:\Codigos\MMQ_MSC_Icaro\venv\Lib\site-packages\sklearn\pipeline.py", line 649, in fit
    self

In [ ]:
evaluate_and_record('MLP Otimo', best_mlp_model, X_train_scaled, y_train, X_test_scaled, y_test, grid_search_mlp.best_params_)

d:\Codigos\MMQ_MSC_Icaro\venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('mlp', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,12
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value
"mean_ mean_: ndarray of shape (n_features,) or NoneThe mean value for each feature in the training set.Equal to ``None`` when ``with_mean=False`` and ``with_std=False``.","ndarray[float64](12,)","[ 0., 0., 0.,...,-0.,-0., 0.]"


# GridSearch Details

In [ ]:
record_gridsearch_details(grid_search_ridge, "Ridge GridSearch")
record_gridsearch_details(grid_search_lasso, "Lasso GridSearch")
record_gridsearch_details(grid_search_rf, "Random Forest GridSearch")
record_gridsearch_details(grid_search_xgb, "XGBoost GridSearch")
record_gridsearch_details(grid_search_knn, "KNN GridSearch")
record_gridsearch_details(grid_search_svr, "SVR GridSearch")
record_gridsearch_details(grid_search_mlp, "Neural Network") 

In [ ]:
results_df

,Modelo,R2 Treino,R2 Teste,RMSE Treino,RMSE Teste,MAE Treino,MAE Teste,Tempo (segundos),Hyperparametros,Tempo (s),Detalhes
0,Regressao Linear,0.613189,4.298225e-01,1278.083951,1445.913293,846.477672,836.380280,0.020213,N/A,NaN,NaN
1,Ridge Otimo,0.612870,4.362528e-01,1278.610278,1437.736842,849.050044,832.471609,0.001878,100.0,NaN,NaN
2,Lasso Otimo,0.598987,4.272652e-01,1301.336058,1449.152169,854.872587,841.330487,0.008264,100.0,NaN,NaN
3,Random Forest Otimo,0.977058,7.924898e-01,311.263984,872.282099,187.760357,598.466417,51.327934,"{'bootstrap': True, 'min_samples_leaf': 3, 'mi...",NaN,NaN
4,Gradient Boosting Otimo,0.986222,8.101815e-01,241.216399,834.269644,170.978050,571.463563,6.734816,"{'learning_rate': 0.0025, 'max_depth': 9, 'n_e...",NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1945,Neural Network - Combinacao 158,NaN,-3.303074e+06,NaN,NaN,NaN,NaN,NaN,"{'mlp__activation': 'tanh', 'mlp__alpha': 0.15...",26.413243,Combin. Testada (Rank: 116)
1946,Neural Network - Combinacao 159,NaN,-2.089002e+06,NaN,NaN,NaN,NaN,NaN,"{'mlp__activation': 'tanh', 'mlp__alpha': 0.15...",374.487276,Combin. Testada (Rank: 39)
1947,Neural Network - Combinacao 160,NaN,-2.519430e+06,NaN,NaN,NaN,NaN,NaN,"{'mlp__activation': 'tanh', 'mlp__alpha': 0.15...",372.273142,Combin. Testada (Rank: 91)
1948,Neural Network - Combinacao 161,NaN,-3.127000e+06,NaN,NaN,NaN,NaN,NaN,"{'mlp__activation': 'tanh', 'mlp__alpha': 0.15...",12.053844,Combin. Testada (Rank: 114)


In [ ]:
results_df.to_csv('model_comparison_results.csv', index=False)